In [0]:
%run ../functions/functions

In [0]:
# Nome do banco de dados onde a tabela será criada ou utilizada
database_name = "dimensao"

# Nome da tabela de destino
table_name = "dm_socios"

# Caminho completo da tabela no formato database.tabela
target_path = f"{database_name}.{table_name}"

# Nome da chave primária da tabela
pk = "SK_SOCIOS"

In [0]:
# Caminho para a tabela Delta de sócios consolidada na camada Silver
silver_path_s = f"abfss://silver@stgbbb.dfs.core.windows.net/cnpj/SOCIOS_CONSOLIDADA/"

# Caminho para a tabela Delta de qualificações consolidadas na camada Silver
silver_path_q = f"abfss://silver@stgbbb.dfs.core.windows.net/cnpj/QUALIFICACOES_CONSOLIDADA/"

In [0]:
# Lê a tabela Delta de sócios a partir do caminho silver
df_s = spark.read.format("delta").load(silver_path_s)
# Lê a tabela Delta de qualificações a partir do caminho silver
df_q = spark.read.format("delta").load(silver_path_q)

In [0]:
# Cria uma view temporária chamada "df_socios" a partir do DataFrame df_s
df_s.createOrReplaceTempView("df_socios")

# Cria uma view temporária chamada "df_qualificacao" a partir do DataFrame df_q
df_q.createOrReplaceTempView("df_qualificacao")

In [0]:
# Consulta SQL para selecionar e enriquecer dados de sócios com descrições de qualificações
query = """SELECT 
  s.SK_SOCIOS,  -- Chave primária dos sócios
  s.cnpj_basico as sk_cnpj_empresas_socios,  -- Chave estrangeira para empresas
  s.identificador_socio,  -- Identificador do sócio
  s.nome_socio,  -- Nome do sócio
  s.documento_cpf_cnpj,  -- Documento do sócio (CPF ou CNPJ)
  s.qualificacao_socio,  -- Código da qualificação do sócio
  q.descricao_qualificacao as qualificacao_socio_desc,  -- Descrição da qualificação do sócio
  s.data_entrada_sociedade,  -- Data de entrada na sociedade
  s.pais,  -- País do sócio
  s.cpf_representante_legal,  -- CPF do representante legal
  s.nome_representante,  -- Nome do representante legal
  s.qualificacao_representante_legal,  -- Código da qualificação do representante legal
  c.descricao_qualificacao as qualificacao_representante_legal_desc,  -- Descrição da qualificação do representante legal
  s.faixa_etaria,  -- Faixa etária do sócio
  s.dt_ingestao  -- Data de ingestão do registro
FROM df_socios as s 
  join df_qualificacao as q on s.qualificacao_socio = q.codigo_qualificacao  -- Join para obter descrição da qualificação do sócio
  join df_qualificacao as c on s.qualificacao_representante_legal = c.codigo_qualificacao;"""  -- Join para obter descrição da qualificação do representante legal

In [0]:
# Executa a consulta SQL definida na variável 'query' e armazena o resultado no DataFrame 'df_join'
df_join = spark.sql(query)

In [0]:
# Adiciona uma coluna "is_socio" ao DataFrame df_join.
# A coluna recebe valor 1 se "qualificacao_socio_desc" contém "socio" (ignorando maiúsculas/minúsculas), caso contrário recebe 0.
df_final = df_join.withColumn(
    "is_socio",
    F.when(
        F.lower(
            F.col("qualificacao_socio_desc")
        ).contains("socio"),
        1
    ).otherwise(0)
)

In [0]:
# Cria o database 'dimensao' caso não exista
spark.sql(f"CREATE DATABASE IF NOT EXISTS dimensao")

In [0]:
# Salva o DataFrame df_final como uma tabela Hive no caminho especificado em target_path,
# utilizando a coluna pk como chave primária.
save_hive_table(df_final, target_path, pk)